In [1]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================
df_puskesmas = pd.read_csv('jmlh_psksms_brdsrkn_plynn_rwt_np_dn_nn_rwt_np.csv')

df_puskesmas

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,PELAYANAN RAWAT INAP,15,UNIT,2018
1,1,21,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,PELAYANAN NON RAWAT INAP,9,UNIT,2018
2,2,32,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,PELAYANAN NON RAWAT INAP,12,UNIT,2018
3,2,42,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,PELAYANAN RAWAT INAP,19,UNIT,2018
4,3,53,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018,PELAYANAN RAWAT INAP,19,UNIT,2018
...,...,...,...,...,...,...,...,...,...,...,...
527,264,528264,35,JAWA TIMUR,3577,KOTA MADIUN,2024,PELAYANAN NON RAWAT INAP,6,UNIT,2024
528,265,529265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,PELAYANAN RAWAT INAP,23,UNIT,2024
529,265,530265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,PELAYANAN NON RAWAT INAP,40,UNIT,2024
530,266,531266,35,JAWA TIMUR,3579,KOTA BATU,2024,PELAYANAN NON RAWAT INAP,2,UNIT,2024


DATA CLEANING

In [2]:
#===============================================================================
# DATA CLEANING
#===============================================================================

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df_puskesmas.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_index',
    'kode_provinsi',
    'kode_kabupaten_kota',
    'periode_update'
]

df_puskesmas[kolom_string] = df_puskesmas[kolom_string].astype(str)

df_puskesmas.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df_puskesmas['nama_kabupaten_kota'] = (
    df_puskesmas['nama_kabupaten_kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df_puskesmas['nama_kabupaten_kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df_puskesmas.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df_puskesmas = df_puskesmas.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# 5. CEK MISSING VALUE
# -------------------------
missing_value = df_puskesmas.isnull().sum()

print("\n4.Cek Missing Value")

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())

    kolom_kategori = 'kategori'
    kolom_numerik = df_puskesmas.select_dtypes(include='number').columns

    for kolom in kolom_numerik:
        if df_puskesmas[kolom].isna().sum() > 0:
            df_puskesmas[kolom] = df_puskesmas.groupby(kolom_kategori)[kolom].transform(
                lambda x: x.fillna(x.median())
            )

    print("Berhasil ditangani sesuai kategori")

else:
    print("Tidak ada missing value")


# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df_puskesmas.select_dtypes(include='number').columns

print("kolom numerik:", kolom_numerik)
print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df_puskesmas[kolom].quantile(0.25)
    Q3 = df_puskesmas[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df_puskesmas[
        (df_puskesmas[kolom] < batas_bawah) |
        (df_puskesmas[kolom] > batas_atas)
    ]


    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532 entries, 0 to 531
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   532 non-null    int64 
 1   id_index             532 non-null    int64 
 2   kode_provinsi        532 non-null    int64 
 3   nama_provinsi        532 non-null    object
 4   kode_kabupaten_kota  532 non-null    int64 
 5   nama_kabupaten_kota  532 non-null    object
 6   periode_update       532 non-null    int64 
 7   kategori             532 non-null    object
 8   jumlah               532 non-null    int64 
 9   satuan               532 non-null    object
 10  tahun                532 non-null    int64 
dtypes: int64(7), object(4)
memory usage: 45.8+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532 entries, 0 to 531
Data columns (total 11 columns):
 #   Column          

TRANSFORMASI DATA

In [3]:
# ==============================================================================
# PIVOT JUMLAH PELAYANAN RAWAT INAP DAN NON RAWAT INAP PER KABUPATEN/KOTA PER TAHUN
# ==============================================================================

# -------------------------
# PIVOT JUMLAH PELAYANAN RAWAT INAP DAN NON RAWAT INAP
# -------------------------
df_puskesmas_tahun_kab = df_puskesmas.pivot_table(
    index=[
        'kode_provinsi',
        'nama_provinsi',
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun'
    ],
    columns='kategori',
    values='jumlah',
    aggfunc='sum'
).reset_index()

# Rename kolom hasil pivot
df_puskesmas_tahun_kab = df_puskesmas_tahun_kab.rename(columns={
    'PELAYANAN RAWAT INAP': 'PELAYANAN_RAWAT_INAP',
    'PELAYANAN NON RAWAT INAP': 'PELAYANAN_NON_RAWAT_INAP'
})

df_puskesmas_tahun_kab = df_puskesmas_tahun_kab.sort_values(
    ['tahun', 'kode_kabupaten_kota'],
    ascending=[True, True]
).reset_index(drop=True)

df_puskesmas_tahun_kab


kategori,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,PELAYANAN_NON_RAWAT_INAP,PELAYANAN_RAWAT_INAP
0,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,9,15
1,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,12,19
2,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018,3,19
3,35,JAWA TIMUR,3504,KABUPATEN TULUNGAGUNG,2018,14,18
4,35,JAWA TIMUR,3505,KABUPATEN BLITAR,2018,6,18
...,...,...,...,...,...,...,...
261,35,JAWA TIMUR,3575,KOTA PASURUAN,2024,8,0
262,35,JAWA TIMUR,3576,KOTA MOJOKERTO,2024,6,0
263,35,JAWA TIMUR,3577,KOTA MADIUN,2024,6,0
264,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,40,23


SIMPAN DATA

In [4]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_puskesmas_tahun_kab.to_csv('data_jumlah_puskesmas.csv', index=False)

In [5]:
print(df_puskesmas_tahun_kab.columns.tolist())

['kode_provinsi', 'nama_provinsi', 'kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'PELAYANAN_NON_RAWAT_INAP', 'PELAYANAN_RAWAT_INAP']
